# 07 · CD en el punto de diseño del compensador (CDC)

**Objetivo.** Mostrar qué entrega el emulador de dispersión cromática en el punto de diseño del grupo del compensador (60 GBd, 72 GSa/s, 200 km de SSMF, OSNR 20 dB):

- que el bloque `ChromaticDispersion` reproduce la CD ideal con esos parámetros, con β₂ y β₃;
- cómo queda la señal a la salida del emulador (sin ruido: el ruido del enlace lo agrega después el bloque AWGN);
- hasta qué distancia el emulador sigue siendo fiel a 72 GSa/s;
- qué tiene que saber el CDC para usar los vectores del emulador (convención de signo, retardo, orden de la cadena).

El compensador usa otro algoritmo: **esta notebook no lo emula**. Solo toma su punto de diseño para mostrar los resultados del emulador.

In [ ]:
# Recarga automática: si se edita el código de emulador/, la notebook usa la versión nueva
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import resample_poly, welch   # cambio de tasa (polifásico) y espectro de Welch

from emulador import plots                        # estilo y guardado de figuras del repo
from emulador.blocks import ChromaticDispersion   # bloque de CD (overlap-save), el del hardware
from emulador.cd import betas, cd_N, cd_R, cd_reference, cd_summary
from emulador.config import C, D_si, S_si         # constante c y conversión de unidades a SI
from emulador.metrics import evm_opt, group_delay, nrmse
from emulador.stimuli import map_gray, prbs       # PRBS y mapeo Gray del emulador

plots.use_style()

## 1. Punto de diseño del grupo del CDC

| Bloque | Parámetro | Valor |
|---|---|---|
| Señal | Tasa de símbolos | 60 GBd |
| | Frecuencia de muestreo | 72 GSa/s (sobremuestreo 1,2) |
| | Modulación | 16-QAM Gray |
| | Largo | 10 000 símbolos |
| | Conformador | coseno realzado, roll-off 0,1, 300 símbolos |
| Canal | Fibra | 200 km, D = 17 ps/(nm·km), λ = 1550 nm |
| | Pendiente de dispersión | S = 0,057 ps/(nm²·km) (β₃; no figura en la captura, pero el grupo presentó resultados con β₃) |
| | Ruido | OSNR 20 dB |
| CDC (referencia) | Ventana FFT / avance | 384 / 192 muestras (algoritmo LOSF del grupo, no se emula acá) |

**Diferencias con el escenario del emulador.**
- 1,2 muestras por símbolo: el `Scenario` del emulador solo admite 1, 2, 4, 8 o 16 sps, pero el bloque de CD recibe $f_s$ directamente y no depende de eso. La señal se genera acá (sección 3).
- Coseno realzado (RC) completo en el transmisor, no raíz (RRC): la señal ya cumple Nyquist y el receptor de esta notebook muestrea sin filtro adaptado.
- β₃: se usa S = 0,057 ps/(nm²·km), el valor típico de SSMF (parámetro `S_PS`, ajustable si el grupo usó otro).

## 2. Parámetros y magnitudes derivadas

In [ ]:
# --- Punto de diseño del grupo del CDC (captura de su presentación) ---
RS = 60e9                          # tasa de símbolos [Bd]
FS = 72e9                          # frecuencia de muestreo [Sa/s]
OS = FS / RS                       # 1,2
ROLLOFF, SPAN = 0.1, 300           # RC del transmisor
N_SYM = 10_000                     # largo de la secuencia [símbolos]
D_PS, LAM_NM, Z_KM = 17.0, 1550.0, 200.0   # fibra: D [ps/(nm·km)], λ [nm], largo [km]
S_PS = 0.057                       # pendiente de dispersión [ps/(nm²·km)] -> β₃
B_CDC = 384                        # ventana FFT del CDC (solo como referencia; su algoritmo no se emula)

# --- Paso a SI (el cálculo interno del emulador es todo en SI) ---
D, lam, z = D_si(D_PS), LAM_NM / 1e9, Z_KM * 1e3  # parametro de dispersión [s/m²] -> β₂, longitud de onda [m], distancia [m]
S = S_si(S_PS)                      # pendiente de dispersión [s/m³] -> β₃
R = cd_R(FS, D, lam, z)             # largo de la rta al impulso de la CD (muestras)
N_EMU = cd_N(FS, D, lam, z)         # bloque del overlap-save del emulador: menor potencia de 2 >= R (mín. 128)


class _Esc:  # cd_summary solo usa fs, Rs y wavelength
    # (Scenario no admite 1,2 sps, así que se le pasa un objeto con esos tres datos)
    fs, Rs, wavelength = FS, RS, lam


s = cd_summary(_Esc, D, z, S)       # magnitudes derivadas: β₂, β₃, R, N, dispersión acumulada, ...
edge = (1 + ROLLOFF) * RS / 2       # borde de la banda de la señal [Hz]
spread_ps = abs(s["gd_slope_ps_GHz"]) * edge / 1e9   # retardo de grupo en el borde de la banda [ps]
print(f"sobremuestreo = {OS:.1f} muestras/símbolo")
print(f"β₂ = {s['beta2_ps2_km']:.2f} ps²/km, β₃ = {s['beta3_ps3_km']:.3f} ps³/km")
print(f"retardo de grupo en el borde de la señal (±{edge / 1e9:.0f} GHz): ±{spread_ps:.0f} ps = ±{spread_ps * 1e-12 * RS:.0f} símbolos")
print(f"R = {R} muestras  (chirp de la CD a {FS / 1e9:.0f} GSa/s y {Z_KM:.0f} km)")
# el overlap-save necesita N >= R: si el chirp es más largo que el bloque, hay aliasing temporal
print(f"emulador: N = {N_EMU} (FFT de {2 * N_EMU}), retardo de la salida cruda = N/2 = {N_EMU // 2} muestras "
      f"= {N_EMU / 2 / FS * 1e12:.0f} ps")
print(f"referencia: la ventana del CDC (B = {B_CDC}) cubre un chirp de hasta {B_CDC // 2 + 1} muestras; R = {R}")

## 3. Señal: 16-QAM Gray con coseno realzado a 1,2 sps

Los símbolos salen de la PRBS-15 del emulador con mapeo Gray (potencia media 1; en la presentación del CDC los niveles no están normalizados, ±1 y ±3, lo que solo cambia la escala). Se conforman con un RC de roll-off 0,1 y 300 símbolos a 6 sps y se toma una de cada 5 muestras: 6/5 = 1,2 sps. No hay aliasing porque la señal ocupa $\pm(1+\rho)R_s/2$ = ±33 GHz y Nyquist está en ±36 GHz.

El receptor de esta notebook interpola por 5 (vuelve a 6 sps) y toma una muestra por símbolo; la EVM se mide con ganancia compleja óptima (`metrics.evm_opt`), descartando 200 símbolos en cada borde.

In [ ]:
# La señal se arma a 6 sps y se pasa a 1,2 sps tomando 1 de cada 5 muestras (6/5 = 1,2)
UP, DOWN = 6, 5          # 6 sps -> /5 -> 1,2 sps
DISCARD = 200            # símbolos descartados por borde en la EVM


def rc_taps(beta, span, sps):
    # Coseno realzado (no raíz): h(t) = sinc(t)·cos(πβt) / (1 - (2βt)²), t en símbolos
    t = np.arange(-span * sps // 2, span * sps // 2 + 1) / sps   # span·sps + 1 taps, centrado en t = 0
    h = np.sinc(t) * np.cos(np.pi * beta * t)
    den = 1 - (2 * beta * t) ** 2
    sing = np.isclose(den, 0)            # en t = ±1/(2β) la fórmula es 0/0
    h[~sing] /= den[~sing]
    h[sing] = np.pi / 4 * np.sinc(1 / (2 * beta))   # valor límite en esos puntos
    return h


def rx_symbols(y):
    # 1,2 sps -> 6 sps (interpolación polifásica) y una muestra por símbolo
    # (el RC cumple Nyquist: en los instantes k·T no hay ISI, no hace falta filtro adaptado)
    return resample_poly(y, DOWN, 1)[: N_SYM * UP][::UP]


# --- Transmisor: PRBS-15 -> Gray 16-QAM -> RC a 6 sps -> 1,2 sps ---
sym = map_gray(prbs(15, 4 * N_SYM), "qam16")      # 4 bits por símbolo, potencia media 1
up = np.zeros(N_SYM * UP, complex)
up[::UP] = sym                                     # un símbolo cada 6 muestras (ceros en el medio)
x6 = np.convolve(up, rc_taps(ROLLOFF, SPAN, UP), mode="same")   # conformación RC
x = x6[::DOWN]                                     # 6 sps -> 1,2 sps (sin aliasing: banda ±33 < ±36 GHz)
x = x / np.sqrt(np.mean(np.abs(x) ** 2))           # potencia media 1
evm_b2b = evm_opt(rx_symbols(x), sym, DISCARD)     # piso de la cadena sin canal
print(f"{x.size} muestras; EVM back-to-back = {100 * evm_b2b:.2f} %")

## 4. CD de 200 km con el emulador

Se aplica el bloque `ChromaticDispersion` con β₂ y β₃ y el N automático del emulador, y se compara con la referencia ideal (`cd_reference`, una FFT de toda la señal, sin bloques). Se muestra solo la CD, sin ruido: en la cadena del emulador el ruido lo agrega después el bloque AWGN (primero el canal, después el ruido).

In [ ]:
# --- Canal: CD de 200 km con β₂ y β₃ (bloque del emulador, N automático) ---
blk = ChromaticDispersion(fs=FS, D_ps_nm_km=D_PS, wavelength_nm=LAM_NM, length_km=Z_KM, S_ps_nm2_km=S_PS)
y_cd = blk(x)                                      # salida alineada (descuenta el retardo N/2)
ref = cd_reference(x, FS, D, lam, z, S=S)          # referencia ideal: una FFT de toda la señal
err_cd = nrmse(y_cd[blk.N : -blk.N], ref[blk.N : -blk.N])   # sin los bordes (transitorios)

evm_cd = evm_opt(rx_symbols(y_cd), sym, DISCARD)       # sin compensar: la señal queda destruida
print(f"N = {blk.N}: NRMSE contra la referencia = {err_cd:.1e}")
print(f"EVM después de la CD (sin compensar) = {100 * evm_cd:.1f} %")

# --- Figura: salida del emulador ---
fig, axs = plt.subplots(3, 1, figsize=(9, 17), gridspec_kw={"height_ratios": [0.7, 1, 1]})
# (a) espectro: la CD es pasa-todo, la magnitud no cambia.
# La entrada se dibuja al final, con trazo discontinuo, para que no quede tapada por la salida.
for sig_, lab, sty in ((y_cd, "salida CD 200 km", dict(color="C1", lw=1)),
                       (x, "entrada", dict(color="k", lw=1.6, ls="--"))):
    f, p = welch(sig_, fs=FS, nperseg=2048, return_onesided=False)
    o = np.argsort(f)                              # ordena de -fs/2 a fs/2 para graficar
    axs[0].plot(f[o] / 1e9, 10 * np.log10(p[o] / p.max()), label=lab, **sty)
for sgn in (-1, 1):
    axs[0].axvline(sgn * edge / 1e9, ls="--", color="k", lw=0.8)    # borde de la señal (±33 GHz)
axs[0].set_ylim(-80, 3)
axs[0].set_xlabel("f [GHz]")
axs[0].set_title("Espectro (PSD normalizada) [dB]")
axs[0].legend(fontsize=9, loc="lower center")
# (b)-(c) constelaciones: entrada y salida de la CD
for ax, sig_, title in ((axs[1], x, "Entrada"), (axs[2], y_cd, "CD 200 km")):
    s_rx = rx_symbols(sig_)[DISCARD:-DISCARD]
    ref_s = sym[DISCARD:-DISCARD]
    g = np.vdot(s_rx, ref_s) / np.vdot(s_rx, s_rx)        # ganancia compleja óptima, como evm_opt
    e = evm_opt(rx_symbols(sig_), sym, DISCARD)
    plots.constellation(ax, g * s_rx, ref=ref_s, title=f"{title} · EVM {100 * e:.1f} %")
plots.save_fig(fig, "07_salida_emulador")

**Qué muestra la figura.** Dentro de la banda, el espectro de la salida coincide con el de la entrada (las curvas se superponen): la CD es un filtro pasa-todo y no cambia la potencia de ninguna frecuencia. Cerca de ±36 GHz la salida sube a unos −35 dB: es el error del método por bloques, que se concentra cerca de Nyquist (decisión 21), y queda fuera de la banda de la señal. Toda la distorsión de la CD está en la fase: la constelación pasa de limpia a una nube sin estructura (el retardo de grupo llega a ±0,9 ns en el borde de la banda, así que cada símbolo recibido mezcla unos ±54 vecinos). Esa nube es lo que recibe el CDC.

## 5. La CD que aplica el emulador: retardo de grupo

Se mide el retardo de grupo de la respuesta del emulador (un impulso a través del bloque) y se compara con la teoría: $\tau(f) = \beta_2 z\,\omega + \tfrac{\beta_3 z}{2}\,\omega^2$, con $\omega = 2\pi f$. Es la forma de verificar que el emulador aplica la dispersión correcta, con el signo de la convención del proyecto.

In [ ]:
# Impulso a través del emulador (salida alineada) -> H(f) -> retardo de grupo
n_imp = 8 * blk.N
imp = np.zeros(n_imp, complex)
imp[n_imp // 2] = 1.0                              # impulso en el centro
h_emu = blk(imp)
f_gd, tau = group_delay(np.fft.fft(np.roll(h_emu, -n_imp // 2)), FS)   # respuesta centrada en n = 0
b2, b3 = betas(D, lam, S)
w = 2 * np.pi * f_gd
tau_th = b2 * z * w + b3 * z / 2 * w**2            # teoría con β₂ y β₃
band = np.abs(f_gd) <= edge                        # banda de la señal (±33 GHz)
rms_tau = np.sqrt(np.mean((tau[band] - tau_th[band]) ** 2))    # error rms en la banda
slope_rel = np.polyfit(w[band], tau[band], 1)[0] / (b2 * z) - 1   # pendiente medida contra β₂·z

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(f_gd[band] / 1e9, tau[band] / 1e-12, label="emulador (N automático)")
ax.plot(f_gd[band] / 1e9, tau_th[band] / 1e-12, "--", label="teoría (β₂ + β₃)")
ax.set_xlabel("f [GHz]")
ax.set_ylabel("τ(f) [ps]")
ax.set_title(f"Retardo de grupo de la CD, {Z_KM:.0f} km, {FS / 1e9:.0f} GSa/s")
ax.legend()
plots.save_fig(fig, "07_retardo_grupo")
print(f"retardo de grupo en ±{edge / 1e9:.0f} GHz: {tau_th[band][0] / 1e-12:+.0f} … {tau_th[band][-1] / 1e-12:+.0f} ps")
print(f"emulador: pendiente {100 * slope_rel:+.3f} % de β₂·z; error rms en la banda = {rms_tau / 1e-12:.1f} ps")
print(f"aporte de β₃ en el borde de la banda: {b3 * z / 2 * (2 * np.pi * edge) ** 2 / 1e-12:.2f} ps")

**Qué muestra la figura.** El retardo de grupo del emulador cae sobre la curva teórica: una recta de pendiente negativa (con $D > 0$ las frecuencias altas llegan antes), de +0,9 a −0,9 ns entre los bordes de la banda; esa diferencia es lo que ensancha cada símbolo sobre sus vecinos. La pendiente coincide con la teórica dentro de 1e-4. La pequeña ondulación cerca de los bordes es el efecto de truncar el chirp al bloque de N muestras (el mismo error de método que da el NRMSE de la sección 4). A 200 km, β₃ aporta apenas ~0,5 ps en el borde de la banda: no se distingue a esta escala (está validado en los tests y en el notebook 04).

## 6. Alcance del emulador a 72 GSa/s

¿Hasta qué distancia el emulador sigue siendo fiel con este muestreo? Se barre z con el N automático (`cd_N`, hasta el máximo provisorio del hardware, N = 2048) y se mide el NRMSE contra la referencia ideal. Como referencia, se marca el chirp que cubre la ventana del CDC (B = 384).

In [ ]:
Z_SWEEP = np.arange(50, 2001, 50)   # distancias del barrido [km]
R_z, N_z, nrmse_z = [], [], []
for zk in Z_SWEEP:
    zz = zk * 1e3
    R_z.append(cd_R(FS, D, lam, zz))
    b_ = ChromaticDispersion(fs=FS, D_ps_nm_km=D_PS, wavelength_nm=LAM_NM, length_km=zk, S_ps_nm2_km=S_PS)
    N_z.append(b_.N)
    r_ = cd_reference(x, FS, D, lam, zz, S=S)      # canal ideal (verdad física)
    nrmse_z.append(nrmse(b_(x)[b_.N : -b_.N], r_[b_.N : -b_.N]))
z_cdc = max(zk for zk in range(1, 3000) if cd_R(FS, D, lam, zk * 1e3) <= B_CDC // 2 + 1)   # alcance de su ventana

fig, axs = plt.subplots(2, 1, figsize=(10, 9), sharex=True)
axs[0].semilogy(Z_SWEEP, nrmse_z, "o-")
axs[0].axhline(1e-2, ls=":", color="k")            # criterio del emulador (decisión 21)
axs[0].set_title("Emulador (N automático) contra la referencia: NRMSE")
axs[1].plot(Z_SWEEP, R_z, "o-", ms=3, label="R (largo del chirp)")
axs[1].step(Z_SWEEP, N_z, where="mid", label="N del emulador")
axs[1].axhline(B_CDC // 2 + 1, ls="--", color="C2", label=f"chirp que cubre la ventana del CDC (B = {B_CDC})")
axs[1].set_title("Largo del chirp y tamaño de bloque [muestras]")
axs[1].legend(fontsize=8)
axs[0].axvline(Z_KM, ls=":", color="C1")
axs[1].axvline(Z_KM, ls=":", color="C1")
axs[-1].set_xlabel("z [km]")                       # eje compartido: etiqueta solo abajo
plots.save_fig(fig, "07_alcance_emulador")

print(f"NRMSE máximo del emulador entre {Z_SWEEP[0]} y {Z_SWEEP[-1]} km: {max(nrmse_z):.1e}")
print(f"la ventana del CDC (B = {B_CDC}) cubre el chirp hasta {z_cdc} km a {FS / 1e9:.0f} GSa/s")
for zk in (100, 200, 500, 1000, 2000):
    i = list(Z_SWEEP).index(zk)
    print(f"{zk:5d} km: R = {R_z[i]:5d}, N = {N_z[i]:5d}, NRMSE = {nrmse_z[i]:.1e}")

**Qué muestra la figura.** A la izquierda, el emulador se mantiene por debajo del criterio de 1e-2 en todo el rango, porque `cd_N` agranda el bloque a medida que crece el chirp. La forma de diente de sierra es la del método: el error crece a medida que R se acerca a N y cae cuando N se duplica (en 182, 363, 725 y 1450 km). A la derecha, el largo del chirp R crece linealmente con la distancia y N sigue a R en potencias de 2; la línea de trazos marca hasta dónde llega la ventana del CDC. Los 200 km del escenario (línea punteada) quedan dentro, con margen.

## 7. Interfaz con el CDC

Lo que el grupo del CDC tiene que saber para usar los vectores del emulador:

- **Convención de signo.** El emulador aplica $H(f) = \exp(+j\pi D\lambda^2 z f^2/c)$ (con β₃ opcional), la ecuación (1) de Jiang, Xie y Chen (JLT, 2026). Para compensarlo hay que aplicar el **conjugado**, $H^*$ (también para β₃). Si el CDC usa la convención opuesta, duplica la dispersión en lugar de anularla.
- **OptiCommPy usa la convención conjugada** (la de Agrawal). Verificado (2026-10-20): su `linearFiberChannel` coincide con el emulador aplicado sobre la señal conjugada (sin β₃: con −D), y su compensador `edc` con D = +17 duplica la dispersión de los vectores del emulador; con D = −17 los compensa. El emulador no depende de OptiCommPy, para ser una verificación independiente.
- **Retardo.** `process()` devuelve la salida alineada con la entrada. `process_hw()` devuelve la salida cruda, como el hardware, retrasada N/2 muestras (128 con N = 256 a 200 km).
- **Orden de la cadena.** Primero el canal, después el ruido (bloque AWGN).
- **Definición de ruido.** Conviene acordar si los 20 dB son OSNR (en 12,5 GHz) o SNR por símbolo: cambian el ruido en 3,8 dB a 60 GBd.

In [ ]:
# Retardo de la salida cruda: el centro de un impulso se corre N/2 muestras
n0 = 4 * blk.N
imp = np.zeros(8 * blk.N, complex)
imp[n0] = 1.0
# el chirp tiene módulo casi constante: el centro se mide con el centro de masa de |y|²
centroid = lambda y_: float(np.sum(np.arange(y_.size) * np.abs(y_) ** 2) / np.sum(np.abs(y_) ** 2))  # noqa: E731
delay_hw = centroid(blk.process_hw(imp)) - n0
delay_al = centroid(blk(imp)) - n0
print(f"delay_samples = {blk.delay_samples}; retardo medido: process_hw {delay_hw:.2f} muestras, process {delay_al:.2f} muestras")

## 8. Validación

In [ ]:
# Los criterios numéricos de la notebook: si algo cambia, falla acá
assert R == 141 and N_EMU == 256 == blk.N            # chirp de 200 km y bloque del emulador
assert evm_b2b < 0.005                               # piso de la cadena sin canal
assert err_cd < 1e-2                                 # el emulador sigue a la CD ideal
assert evm_cd > 0.5                                  # sin compensar, la señal queda destruida
assert abs(slope_rel) < 1e-3 and rms_tau < 15e-12   # retardo de grupo: pendiente de β₂·z y error rms < 15 ps
assert max(nrmse_z) < 1e-2                           # emulador fiel en 50-2000 km a 72 GSa/s
assert z_cdc >= Z_KM                                 # la ventana del CDC cubre el escenario
assert abs(delay_hw - blk.N // 2) < 0.5 and abs(delay_al) < 0.5   # retardo N/2 declarado
print("OK")

## 9. Conclusiones

- **El emulador reproduce la CD en el punto de diseño del CDC.** A 60 GBd, 72 GSa/s y 200 km (con β₃), el chirp ocupa R = 141 muestras; el emulador elige N = 256 (FFT de 512) y sigue a la CD ideal con NRMSE 1,8e-3. El retardo de grupo que aplica tiene la pendiente teórica (−27,2 ps/GHz) con error de 1e-4.
- **Lo que recibe el CDC.** La CD no cambia el espectro en la banda (pasa-todo), pero el retardo de grupo llega a ±0,9 ns en el borde de la banda: cada símbolo mezcla unos ±54 vecinos y la constelación queda destruida (EVM 99,5 %). El ruido del enlace (OSNR 20 dB) lo agrega después el bloque AWGN: primero el canal, después el ruido.
- **Alcance del emulador a 72 GSa/s**: NRMSE ≤ 7,5e-3 entre 50 y 2000 km con el N automático (hasta N = 2048). La ventana del CDC (B = 384) cubre el chirp hasta 274 km, así que los 200 km del escenario tienen margen.
- **Para el CDC**: el emulador sigue la convención de la ec. (1) de Jiang, Xie y Chen (2026); el CDC tiene que usar el conjugado ($H^*$). OptiCommPy usa la convención conjugada: con su `edc`, D va negativo. La salida cruda (`process_hw`) sale retrasada N/2 = 128 muestras.
- **A acordar con el grupo**: la convención de signo, si los 20 dB son OSNR o SNR por símbolo, y el formato de los vectores de prueba.

Figuras en `docs/golden_model/figuras/07_*.png`.